# Find Patents

In [1]:
import pandas as pd
df = pd.read_csv("D:\\New-patents\\Landscaping\\data\\seed_and_antiseed_dataset.csv")
df

C:\Users\Asus\AppData\Roaming\Python\Python39\site-packages\pandas\core\arrays\masked.py:60: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


,Num_patent,Similarity,flag_patent,grp_ml,grp_evo,grp_nlp,grp_speech,grp_vision,grp_kr,grp_planning,grp_hardware
0,7664556,0.090971,1,na,antiseed,na,na,na,na,l1,na
1,6199040,0.089747,1,na,na,l2,l1,na,na,antiseed,na
2,7885968,0.087807,1,na,na,l2,l2,na,l2,na,antiseed
3,5822496,0.087701,1,na,na,l2,na,l1,l1,l1,seed
4,6134344,0.087036,1,l1,l2,l2,l2,seed,l1,l2,l2
...,...,...,...,...,...,...,...,...,...,...,...
72430,5139541,-0.029689,1,na,antiseed,na,na,na,na,na,na
72431,4366272,-0.030128,1,na,na,na,antiseed,na,na,na,na
72432,8552104,-0.030188,1,na,na,na,na,na,na,na,antiseed
72433,4428840,-0.030381,1,na,antiseed,na,na,na,na,na,na


In [2]:
def determine_is_ai(row):
    if 'seed' in row.values:
        return 1
    elif 'antiseed' in row.values:
        return 0
    return pd.NA

# Applying the function row-wise
df['is_ai'] = df.apply(determine_is_ai, axis=1)

In [10]:
df

,Num_patent,Similarity,flag_patent,grp_ml,grp_evo,grp_nlp,grp_speech,grp_vision,grp_kr,grp_planning,grp_hardware,is_ai
0,7664556,0.090971,1,na,antiseed,na,na,na,na,l1,na,0
1,6199040,0.089747,1,na,na,l2,l1,na,na,antiseed,na,0
2,7885968,0.087807,1,na,na,l2,l2,na,l2,na,antiseed,0
3,5822496,0.087701,1,na,na,l2,na,l1,l1,l1,seed,1
4,6134344,0.087036,1,l1,l2,l2,l2,seed,l1,l2,l2,1
...,...,...,...,...,...,...,...,...,...,...,...,...
72430,5139541,-0.029689,1,na,antiseed,na,na,na,na,na,na,0
72431,4366272,-0.030128,1,na,na,na,antiseed,na,na,na,na,0
72432,8552104,-0.030188,1,na,na,na,na,na,na,na,antiseed,0
72433,4428840,-0.030381,1,na,antiseed,na,na,na,na,na,na,0


In [ ]:
num_patents_seed = []
num_patents_antiseed = []

data_seed = df[df['is_ai'] == 1].sample(400)['Num_patent']
data_antiseed = df[df['is_ai'] == 0].sample(600)['Num_patent']
for i in range(len(data_seed)):
    num_patents_seed.append(data_seed.iloc[i])
for i in range(len(data_antiseed)):
    num_patents_antiseed.append(data_antiseed.iloc[i])

In [6]:
num_patents = num_patents_seed + num_patents_antiseed

# load patent texts

In [ ]:
g =open("D:\\New-patents\\Landscaping\\LDA phrases\\data\\fulltext_concat.tsv")
l=g.readline()# header
patent_texts = {}
i=0
while (True):
    l=g.readline()
    s=l.split()
    if len(s) > 10:
        q =2
        n = s[1]
        if (n.isnumeric() == False) or (len(n) < 3):
            continue
        for id in num_patents:
            if id == int(n):
                patent_texts[id] = " ".join(s[q:])
                print(len(patent_texts))
        if len(patent_texts) == 1000:
            break
    

In [14]:
sorted_d = {}
for k in num_patents:
    if k in patent_texts.keys():
        sorted_d[int(k)] = patent_texts[k]


In [25]:
import pickle
f=open('./data/972_samples_seed', 'wb')
pickle.dump(sorted_d,f)
f.close()

# LLM model

In [74]:
import json
import re
from llama_cpp import Llama

def load_llm(model_path: str, n_ctx: int = 16384, n_gpu_layers: int = -1):
    return Llama(model_path=model_path, n_ctx=n_ctx, n_gpu_layers=n_gpu_layers, verbose=False)

def extract_json_robust(text: str) -> dict:
    """Multi-strategy JSON extraction for LLM outputs"""
    # 1. Strip markdown
    cleaned = re.sub(r'^```(?:json)?\s*', '', text)
    cleaned = re.sub(r'\s*```$', '', cleaned).strip()
    
    # 2. Fix Python literals
    cleaned = re.sub(r'\bTrue\b', 'true', cleaned)
    cleaned = re.sub(r'\bFalse\b', 'false', cleaned)
    cleaned = re.sub(r'\bNone\b', 'null', cleaned)
    
    # 3. Close unclosed brackets/braces
    stack = []
    for char in cleaned:
        if char in '{[': stack.append(char)
        elif char == '}': 
            if stack and stack[-1] == '{': stack.pop()
        elif char == ']': 
            if stack and stack[-1] == '[': stack.pop()
    cleaned += '}' * stack.count('{') + ']' * stack.count('[')
    
    # 4. Remove trailing commas
    cleaned = re.sub(r',\s*([\]}])', r'\1', cleaned)
    
    # 5. Try direct parse
    try:
        return json.loads(cleaned)
    except json.JSONDecodeError:
        pass
    
    # 6. Fallback: extract known keys via regex
    keys = ["overall_ai_score", "subdomain_scores", "ai_extraction", "reasoning"]
    fallback = {}
    for k in keys:
        match = re.search(rf'"{k}"\s*:\s*(".*?"|[\d.]+|true|false|null|\{{.*?\}})', cleaned)
        if match:
            val = match.group(1).strip('"')
            fallback[k] = val if val not in ("true", "false", "null") else json.loads(val)
    
    return fallback if fallback else None


def analyze_patent(llm: Llama, patent_text: str) -> dict:
    system_prompt = "You are a precise AI patent analyst. Output ONLY raw JSON. No markdown. No explanations."
    
    user_prompt = f"""You are an AI researcher and patent examiner. Analyze the provided patent text and assess its relationship to Artificial Intelligence (AI) across theoretical, applied, and implementation dimensions.

OUTPUT ONLY RAW JSON. NO MARKDOWN. NO EXPLANATIONS.

{{
  "ai_extraction": "string (inline [AI: ...] markers or 'None found')",
  "reasoning": "string (theoretical vs applied, core vs peripheral)",
  "overall_ai_score": <float 0.0-1.0>,
  "overall_ai_justification": "<1-2 sentences using rubric>",
  "subdomain_scores": {{
    "machine_learning": <float>,
    "computer_vision": <float>,
    "natural_language_processing": <float>,
    "speech": <float>,
    "evolutionary_computation": <float>,
    "ai_hardware": <float>,
    "knowledge_processing": <float>,
    "planning_and_control": <float>
  }},
  "subdomain_justifications": {{
    "machine_learning": "<string>",
    "computer_vision": "<string>",
    "natural_language_processing": "<string>",
    "speech": "<string>",
    "evolutionary_computation": "<string>",
    "ai_hardware": "<string>",
    "knowledge_processing": "<string>",
    "planning_and_control": "<string>"
  }}
}}

SCORING RUBRIC:
• 0.0–0.3: AI is incidental, mentioned once, or used as a generic tool with no technical detail.
• 0.4–0.6: AI is present and described, but peripheral to the invention's novelty (e.g., "AI optimizes X" where X is the core).
• 0.7–0.85: AI is central to the technical solution, with clear implementation details or architectural novelty.
• 0.9–1.0: Core invention is an AI method, model, or system; AI defines the novelty and claims.
Note: Weight Claims heavily. AI described only in examples or background should score lower.

STRICT RULES:
1. EXTRACT ONLY EXACT VERBATIM PHRASES OR COMPLETE SENTENCES. Do not extract isolated words, paraphrase, or summarize.
2. All JSON keys listed above MUST be present. If no AI is found, set overall_ai_score=0.0, and return empty/zero values for lists and scores.
3. Use precise AI terminology: neural networks, training, inference, loss, gradient, embedding, clustering, generative, reinforcement, optimization, pattern recognition, model deployment, API integration, etc.
4. Output MUST be valid JSON. Do not add explanations, markdown, or trailing text after the JSON object.
5. Keep output concise. Do not exceed 1500 tokens.
6. If there aren't any of parameters, Please write null in value of that.

Patent Text:
{patent_text}
"""

    messages = [{"role": "system", "content": system_prompt}, {"role": "user", "content": user_prompt}]
    
    output = llm.create_chat_completion(
        messages=messages,
        max_tokens=4096,
        temperature=0.0,
        top_p=0.9,
        repeat_penalty=1.1,
        response_format={"type": "json_object"}
    )
    
    raw = output["choices"][0]["message"]["content"]
    result = extract_json_robust(raw)
    
    # Validate structure
    required = ["overall_ai_score", "subdomain_scores", "ai_extraction", "reasoning"]
    missing = [k for k in required if k not in (result or {})]
    
    if not result or missing:
        print("\n=== RAW LLM OUTPUT ===")
        print(raw)
        print("\n=== DIAGNOSTICS ===")
        print(f"Length: {len(raw)} chars")
        print(f"Missing keys: {missing}")
        print(f"Starts with: {raw[:50]}")
        print(f"Ends with: ...{raw[-50:]}")
        print(f"Contains '```': {'```' in raw}")
        return {"error": "JSON structure incomplete or invalid", "missing_keys": missing, "raw_length": len(raw)}
    
    return result

In [28]:
# # ================= USAGE =================
MODEL_PATH = "C:\\Users\\Asus\\.lmstudio\\models\\lmstudio-community\\Qwen3.6-35B-A3B-GGUF\\Qwen3.6-35B-A3B-Q4_K_M.gguf"  # Adjust path
PATENT_TEXT = f"""[{sorted_d[5376962][:35000]}]"""
llm = load_llm(MODEL_PATH, n_ctx=16384)
result = analyze_patent(llm, PATENT_TEXT)
print(json.dumps(result, indent=2))

llama_context: n_ctx_seq (16384) < n_ctx_train (262144) -- the full capacity of the model will not be utilized


{
  "ai_extraction": "The present invention is embodied in apparatus and a method which utilizes a neural network to perform a group of independent signal processing functions as a single, combined function. The neural network includes a plurality of input processors which receive the signals to be processed. The network also includes one or more layers of hidden processors and a layer of output processors. ... on-line training algorithm which is based on the defined parameter sets. ... back propagation means, responsive to the cost function values, for modifying each of the programmable weighting factors to minimize the cost function values; and means for determining when the weighting factors have converged to values which allow the neural network to emulate the signal processing function with acceptable error",
  "reasoning": "The invention's core novelty is a system using a neural network to emulate multiple nonlinear video signal processing functions. The claims heavily rely on th

# create dataset

In [ ]:
import pickle
import pandas as pd
f=open("./data/972_samples_seed", "rb")
sorted_d=pickle.load(f)
f.close()

In [30]:
sorted_d

{5376962: '"BACKGROUND OF THE INVENTION The present invention concerns apparatus and a method for processing video signals to implement a plurality of signal processing functions in a single operation, and in particular to apparatus which uses a neural network to perform these functions. In many signal processing systems, functions are applied to data sequentially. That is to say one function is applied to the input data and the output values produced by that function are applied as the input values to the next function. This type of processing is commonly referred to as pipeline processing. An exemplary system which employs pipeline processing is a video camera. One such camera is described in an article by K. A. Parulski et al. entitled ""A Digital Color CCD Imaging System Using Custom VLSI Circuits,"" IEEE Transactions on Consumer Electronics, Vol. 35, No. 3, pp. 382-389, August, 1989. The camera described in this paper employs a CCD sensor which collects samples representing succes

In [34]:
patent_id = []
patent_text = []
ai_extraction = []
reasoning = []
overall_ai_score = []
overall_ai_justification = []
subdomain_scores = []
subdomain_justifications = []
dataset = pd.DataFrame()


In [ ]:
MODEL_PATH = "C:\\Users\\Asus\\.lmstudio\\models\\lmstudio-community\\Qwen3.6-35B-A3B-GGUF\\Qwen3.6-35B-A3B-Q4_K_M.gguf"  # Adjust path
j=0
for id in list(sorted_d.keys()):
    
    
    print(id)
    print(j)
    if j<=287:
        j = j+1
        continue

    text = sorted_d[id]
    if len(text) > 35000:
        ch = len(text)//35000
        for i in range(ch+1):
            if i ==ch:
                text1 = text[(i*35000):]
            else:
                text1 = text[(i*35000):((i+1)*35000)]
                
            PATENT_TEXT = f"""[{text1}]"""
            llm = load_llm(MODEL_PATH, n_ctx=16384)
            result = analyze_patent(llm, PATENT_TEXT)
            a = json.dumps(result, indent=2)
            result_dict = json.loads(a)
            
            patent_id.append(str(id)+f'_{i}')
            patent_text.append(text1)
            ai_extraction.append(result_dict['ai_extraction'])
            reasoning.append(result_dict['reasoning'])
            overall_ai_score.append(result_dict['overall_ai_score'])
            overall_ai_justification.append(result_dict['overall_ai_justification'])
            subdomain_scores.append(result_dict['subdomain_scores'])
            subdomain_justifications.append(result_dict['subdomain_justifications'])
    else:
        PATENT_TEXT = f"""[{text}]"""
        llm = load_llm(MODEL_PATH, n_ctx=16384)
        result = analyze_patent(llm, PATENT_TEXT)
        a = json.dumps(result, indent=2)
        result_dict = json.loads(a)

        patent_id.append(id)
        patent_text.append(text)
        ai_extraction.append(result_dict['ai_extraction'])
        reasoning.append(result_dict['reasoning'])
        overall_ai_score.append(result_dict['overall_ai_score'])
        overall_ai_justification.append(result_dict['overall_ai_justification'])
        subdomain_scores.append(result_dict['subdomain_scores'])
        subdomain_justifications.append(result_dict['subdomain_justifications'])
    j = j+1

In [ ]:

dataset["patent_id"] = patent_id
dataset["patent_text"] = patent_text
dataset["ai_extraction"] = ai_extraction
dataset["reasoning"] = reasoning
dataset["overall_ai_score"] = overall_ai_score
dataset["overall_ai_justification"] = overall_ai_justification
dataset["subdomain_scores"] = subdomain_scores
dataset["subdomain_justifications"] = subdomain_justifications

dataset

,patent_id,patent_text,ai_score,theoretical_aspects,applied_aspects,ai_role,verbatim_ai_phrases,non_ai_exclusions,verification_note
0,8909570,The following patent applications are also inc...,0.85,"[Age-Layered Population Structure (ALPS), elit...",[deployment of selected evolved rules for cont...,core_invention,[{'text': 'The invention relates generally to ...,"[traditional rule-based systems, hardware sens...",All verbatim phrases are extracted exactly as ...
1,8751416,CROSS-REFERENCE TO RELATED APPLICATIONS The pr...,0.75,[generation of a probabilistic model from a de...,[MLA operative to receive data from the VDM an...,core_invention,[{'text': 'a machine learning algorithm (MLA) ...,[dynamic Bayesian Network (DBN) derived from o...,All phrases are extracted verbatim from the pr...
2,8170963,CROSS REFERENCES TO RELATED APPLICATIONS The p...,0.75,[genetic algorithm with crossover and mutation...,[processing detection data using learned featu...,core_invention,[{'text': 'The genetic search method in the re...,[traditional linear combination coefficient es...,All verbatim phrases are extracted exactly fro...
3,8775341_0,FIELD OF THE INVENTION The present invention r...,0.95,[hierarchical stacked neural networks configur...,[training neural network models using gradient...,core_invention,[{'text': 'The present invention relates to th...,"[traditional PID control (not mentioned), rule...",All verbatim phrases are extracted exactly as ...
4,8775341_1,re applied across the entire space of possible...,0.95,[Hierarchical stacked computer neural networks...,"[Detecting aberrant messages, such as email sp...",core_invention,[{'text': 'The present invention relates to th...,"[Traditional PID control mechanisms, Rule-base...",All verbatim phrases are extracted exactly as ...
...,...,...,...,...,...,...,...,...,...
103,7789091,BACKGROUND OF THE INVENTION 1. Field of the In...,0.00,[],[],none,[],"[abrasive coating, flexible screen member, mec...",All phrases exist verbatim in the provided text
104,8435730,FIELD OF THE INVENTION The invention relates t...,0.00,[],[],none,[],"[chemical composition formulation, cryogenic p...",All phrases exist verbatim in the provided text
105,5062968,"""BACKGROUND OF THE INVENTION The apparatus and...",0.00,[],[],none,[],"[mechanical filtration, pressurized gas system...",All phrases exist verbatim in the provided text.
106,8151913,BACKGROUND OF THE INVENTION 1. Field of the In...,0.00,[],[],none,[],"[differential mechanism, gear reduction system...",All phrases exist verbatim in the provided text.


In [ ]:
dataset.to_csv("./data/972_ai_seed_samples_data.csv")

# extract ai related (for low ai score)

In [1]:
import json
import re
from llama_cpp import Llama

def load_llm(model_path: str, n_ctx: int = 16384, n_gpu_layers: int = -1):
    return Llama(model_path=model_path, n_ctx=n_ctx, n_gpu_layers=n_gpu_layers, verbose=False)

def extract_json_robust(text: str) -> dict:
    """Multi-strategy JSON extraction for LLM outputs"""
    # 1. Strip markdown
    cleaned = re.sub(r'^```(?:json)?\s*', '', text)
    cleaned = re.sub(r'\s*```$', '', cleaned).strip()
    
    # 2. Fix Python literals
    cleaned = re.sub(r'\bTrue\b', 'true', cleaned)
    cleaned = re.sub(r'\bFalse\b', 'false', cleaned)
    cleaned = re.sub(r'\bNone\b', 'null', cleaned)
    
    # 3. Close unclosed brackets/braces
    stack = []
    for char in cleaned:
        if char in '{[': stack.append(char)
        elif char == '}': 
            if stack and stack[-1] == '{': stack.pop()
        elif char == ']': 
            if stack and stack[-1] == '[': stack.pop()
    cleaned += '}' * stack.count('{') + ']' * stack.count('[')
    
    # 4. Remove trailing commas
    cleaned = re.sub(r',\s*([\]}])', r'\1', cleaned)
    
    # 5. Try direct parse
    try:
        return json.loads(cleaned)
    except json.JSONDecodeError:
        pass
    
    # 6. Fallback: extract known keys via regex
    keys = ["ai_extraction"]
    fallback = {}
    for k in keys:
        match = re.search(rf'"{k}"\s*:\s*(".*?")', cleaned)
        if match:
            val = match.group(1).strip('"')
            fallback[k] = val
    
    return fallback if fallback else None


def analyze_patent(llm: Llama, patent_text: str) -> dict:
    system_prompt = "You are a precise AI patent analyst. Output ONLY raw JSON. No markdown. No explanations."
    
    user_prompt = f"""You are an AI expert and patent examiner. You are given a patent text and your job is to read it and extract and output all AI-related sentences and paragraphs. Consider Artificial Intelligence (AI) in a broad but precise sense, across theoretical, applied, and implementation dimensions. Take into account old AI (symbolic AI, fuzzy logic) and AI hardware, as well as new and mainstream AI.

EXTRACT ONLY EXACT VERBATIM COMPLETE SENTENCES. Do not extract isolated words, paraphrase, or summarize.

OUTPUT ONLY RAW JSON. NO MARKDOWN. NO EXPLANATIONS.

{{
  "ai_extraction": "string (verbatim complete sentences extracted from the patent, separated by spaces or newlines. If no AI found, write 'None found')"
}}

STRICT RULES:
1. EXTRACT ONLY EXACT VERBATIM COMPLETE SENTENCES into the single 'ai_extraction' field.
2. Output MUST be valid JSON. Do not add explanations, markdown, or trailing text after the JSON object.
3. If no AI found, set ai_extraction="None found".

Patent Text:
{patent_text}
"""

    messages = [{"role": "system", "content": system_prompt}, {"role": "user", "content": user_prompt}]
    
    output = llm.create_chat_completion(
        messages=messages,
        max_tokens=4096,
        temperature=0.0,
        top_p=0.9,
        repeat_penalty=1.1,
        response_format={"type": "json_object"}
    )
    
    raw = output["choices"][0]["message"]["content"]
    result = extract_json_robust(raw)
    
    # Validate structure
    if not result or "ai_extraction" not in result:
        print("\n=== RAW LLM OUTPUT ===")
        print(raw)
        print("\n=== DIAGNOSTICS ===")
        print(f"Length: {len(raw)} chars")
        print(f"Starts with: {raw[:50]}")
        print(f"Ends with: ...{raw[-50:]}")
        print(f"Contains '```': {'```' in raw}")
        return {"error": "JSON structure incomplete or invalid", "raw_length": len(raw)}
    
    return result

In [6]:
import pandas as pd
data = pd.read_csv("./LDA phrases/data/llm_without_chunks.csv")

In [12]:
data = data[data['ai_extraction'] == 'null found']
data = data[data['overall_ai_score'] != 0]

In [13]:
data

,Unnamed: 0.2,Unnamed: 0.1,Unnamed: 0,patent_id,patent_text,ai_extraction,reasoning,overall_ai_score,overall_ai_justification,subdomain_scores,...,their_score_planning,Our_work,our_binary_ml,our_binary_evo,our_binary_nlp,our_binary_speech,our_binary_vision,our_binary_hardware,our_binary_kr,our_binary_planning
2,2,2,2,5568413,"""BACKGROUND OF THE INVENTION The present inven...",null found,The patent describes a fuzzy logic controller ...,0.1,Fuzzy logic is a classical control theory tech...,"{'machine_learning': 0.0, 'computer_vision': 0...",...,0.530319,0.027891,1.0,0.0,0.0,0.0,0.0,1.0,1.0,1.0
6,6,6,6,6678670,FIELD OF THE INVENTION The present invention r...,null found,The patent describes a circuit for implementin...,0.5,AI (neural network) is present as an implement...,"{'machine_learning': 0.4, 'computer_vision': 0...",...,0.999991,0.037521,1.0,0.0,0.0,0.0,0.0,1.0,1.0,1.0
8,8,8,8,5677995,"""The present invention relates to a control me...",null found,The patent describes a control method and circ...,0.2,Fuzzy logic is mentioned as the underlying con...,"{'machine_learning': 0.0, 'computer_vision': 0...",...,0.997792,0.013530,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0
15,17,17,17,9037472,BACKGROUND OF THE INVENTION Many human-to-mach...,null found,The patent describes a system for determining ...,0.2,AI is incidental; the invention relies on stat...,"{'machine_learning': 0.0, 'computer_vision': 0...",...,0.989813,0.005579,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
17,19,19,19,8321417,BACKGROUND OF THE INVENTION 1. Field of the In...,null found,The patent describes a group decision-making t...,0.2,AI is incidental; 'clustering' is mentioned as...,"{'machine_learning': 0.1, 'computer_vision': 0...",...,0.419149,0.006888,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
840,949,951,951,10122868,CROSS-REFERENCE TO RELATED APPLICATION The pre...,null found,The patent describes a method for submitting e...,0.1,"AI is incidental, mentioned once as a generic ...","{'machine_learning': 0.0, 'computer_vision': 0...",...,0.155573,0.004989,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0
842,951,953,953,9743000,BACKGROUND OF THE INVENTION Field of the Inven...,null found,The patent describes a deterministic image pro...,0.1,AI is incidental; the invention uses determini...,"{'machine_learning': 0.0, 'computer_vision': 0...",...,0.000013,0.002438,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
856,967,969,969,5598359,"""TECHNICAL FIELD OF THE INVENTION This inventi...",null found,The patent describes a weather effects generat...,0.1,AI is incidental or absent; the invention focu...,"{'machine_learning': 0.0, 'computer_vision': 0...",...,0.980479,0.016239,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
862,973,975,975,8842026,BACKGROUND Automotive systems are complex syst...,null found,The patent describes a threshold estimation sy...,0.1,AI is incidental; the system relies on determi...,"{'machine_learning': 0.0, 'computer_vision': 0...",...,0.000024,0.004010,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0


In [ ]:
import pickle
MODEL_PATH = "C:\\Users\\Asus\\.lmstudio\\models\\lmstudio-community\\Qwen3.6-35B-A3B-GGUF\\Qwen3.6-35B-A3B-Q4_K_M.gguf"  # Adjust path
# f=open("./list_ai_extract100", "rb")
# list_extraction=pickle.load(f)
# f.close()
list_extraction = {}
j=1
for id in list(data["patent_id"]):
    print(id)
    print(j)
    if id == 7706601:
        continue
    if j > 0:
        text = data[data["patent_id"]==id]['patent_text'].iloc[0]
        # if len(text) > 35000:
        #     ch = len(text)//35000
        #     for i in range(ch+1):
        #         if i ==ch:
        #             text1 = text[(i*35000):]
        #         else:
        #             text1 = text[(i*35000):((i+1)*35000)]
                    
        PATENT_TEXT = f"""[{text}]"""
        llm = load_llm(MODEL_PATH, n_ctx=16384)
        result = analyze_patent(llm, PATENT_TEXT)
        a = json.dumps(result, indent=2)
        result_dict = json.loads(a)
        list_extraction[id] = result_dict['ai_extraction']
        if j%50==0:
            f=open(f'./list_ai_extract{j}', 'wb')
            pickle.dump(list_extraction,f)
            f.close()
    j=j+1       
    #7706601

In [ ]:
f=open('./list_ai_extract', 'wb')
pickle.dump(list_extraction,f)
f.close()

In [2]:
import pickle
f=open("./list_ai_extract", "rb")
list_extraction=pickle.load(f)
f.close()

In [5]:
list_extraction

{5568413: 'The present invention relates to a fuzzy logic controller of the analog type which implements a set of fuzzy rules with which are respectively associated output values which can be expressed as as many polynomials of a same set of input variables, the coefficients of said polynomials having predetermined values (possibly equal to zero) specific to the different rules and each of said rules including at least one condition consisting of a fuzzy relationship between one of said input variables and a reference value. Known fuzzy logic controllers are provided in order to implement fuzzy rules for which the output values are predetermined constants. Such fuzzy rules define an association between a fuzzy subset of an input space and a point in an output space. Typically, such a known fuzzy rule will take the following form: EQU IF x.sub.1 .apprxeq.p.sub.1 AND x.sub.2 .apprxeq.p.sub.2 THEN y=u where p.sub.1 and p.sub.2 are reference values which define the center of a fuzzy domain

In [17]:
for id in list_extraction.keys():
    data.ai_extraction[data.patent_id ==id] = list_extraction[id]

C:\Users\Asus\AppData\Local\Temp\ipykernel_4128\3978518372.py:2: FutureWarning: ChainedAssignmentError: behaviour will change in pandas 3.0!
You are setting values through chained assignment. Currently this works in certain cases, but when using Copy-on-Write (which will become the default behaviour in pandas 3.0) this will never work to update the original DataFrame or Series, because the intermediate object on which we are setting values will behave as a copy.
A typical example is when you are setting values in a column of a DataFrame, like:

df["col"][row_indexer] = value

Use `df.loc[row_indexer, "col"] = values` instead, to perform the assignment in a single step and ensure this keeps updating the original `df`.

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

  data.ai_extraction[data.patent_id ==id] = list_extraction[id]
C:\Users\Asus\AppData\Local\Temp\ipykernel_4128\3978518372.py:2: Set

In [21]:
data.to_csv("./data_919.csv")